In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [2]:
articles = pd.read_parquet("../../../data/gold/articles.parquet")

articles.shape

(105542, 15)

In [3]:
articles["combined_text"] = (
    articles["prod_name"].astype(str) + " " +
    articles["detail_desc"].astype(str)
)

In [4]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=50000
)

text_matrix = tfidf_vectorizer.fit_transform(
    articles["combined_text"]
)

text_matrix.shape

(105542, 50000)

In [5]:
nn_text_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

nn_text_model.fit(text_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [6]:
query_idx = 0

distances, indices = nn_text_model.kneighbors(
    text_matrix[query_idx],
    n_neighbors=20
)

distances, indices

(array([[0.        , 0.        , 0.        , 0.34287333, 0.34287333,
         0.34287333, 0.54873964, 0.56916138, 0.5817938 , 0.58819146,
         0.59747586, 0.59747586, 0.59747586, 0.59747586, 0.59747586,
         0.59747586, 0.59747586, 0.59835391, 0.59835391, 0.60366582]]),
 array([[    0,     1,     2, 10109, 10110, 10111, 77230, 35305, 87915,
         83527, 28591, 28588, 28587, 28586, 28592, 28590, 28589, 72495,
         72496, 62642]]))

In [7]:
articles.iloc[indices[0][:10]][[
    "article_id",
    "product_code",
    "prod_name",
    "colour_group_name",
    "product_type_name"
]]

,article_id,product_code,prod_name,colour_group_name,product_type_name
0,108775015,108775,Strap top,Black,Vest top
1,108775044,108775,Strap top,White,Vest top
2,108775051,108775,Strap top (1),Off White,Vest top
10109,538699001,538699,V-neck strap top,Black,Vest top
10110,538699002,538699,V-neck strap top,Light Beige,Vest top
10111,538699007,538699,V-neck strap top,White,Vest top
77230,788464001,788464,Chloe Dress,Dark Red,Dress
35305,646265001,646265,ES Alexandra strap,Black,Vest top
87915,830508001,830508,ED Strap top 3p,Black,Top
83527,812371001,812371,Strap top 3-pack,Black,Vest top


In [10]:
def get_similar_products_text(article_id, top_k=10, candidate_pool=200):
    matching_indices = articles.index[
        articles["article_id"] == article_id
    ].tolist()

    if not matching_indices:
        raise ValueError(f"article_id bulunamadı: {article_id}")

    query_idx = matching_indices[0]
    query_product_code = articles.loc[query_idx, "product_code"]

    distances, indices = nn_text_model.kneighbors(
        text_matrix[query_idx],
        n_neighbors=candidate_pool
    )

    filtered_neighbors = []
    seen_product_codes = set()

    for idx, dist in zip(indices[0], distances[0]):
        candidate_product_code = articles.loc[idx, "product_code"]

        # Kendisini çıkar
        if idx == query_idx:
            continue

        # Sorgu ürününün farklı renk varyantlarını çıkar
        if candidate_product_code == query_product_code:
            continue

        # Aynı önerilen ürün ailesinin farklı renklerini tekrar gösterme
        if candidate_product_code in seen_product_codes:
            continue

        filtered_neighbors.append((idx, dist))
        seen_product_codes.add(candidate_product_code)

        if len(filtered_neighbors) == top_k:
            break

    neighbor_indices = [idx for idx, _ in filtered_neighbors]
    neighbor_distances = [dist for _, dist in filtered_neighbors]

    display_cols = [
        "article_id",
        "product_code",
        "prod_name",
        "product_type_name",
        "colour_group_name",
        "section_name",
        "garment_group_name"
    ]

    query_product = articles.iloc[[query_idx]][display_cols]

    recommendations = articles.iloc[neighbor_indices][display_cols].copy()
    recommendations["cosine_similarity"] = 1 - pd.Series(
        neighbor_distances,
        index=recommendations.index
    )

    return query_product, recommendations

In [11]:
query_product, recommendations = get_similar_products_text(
    article_id=108775015,
    top_k=10
)

query_product, recommendations

(   article_id  product_code  prod_name product_type_name colour_group_name  \
 0   108775015        108775  Strap top          Vest top             Black   
 
              section_name garment_group_name  
 0  Womens Everyday Basics       Jersey Basic  ,
        article_id  product_code           prod_name product_type_name  \
 10109   538699001        538699    V-neck strap top          Vest top   
 77230   788464001        788464         Chloe Dress             Dress   
 35305   646265001        646265  ES Alexandra strap          Vest top   
 87915   830508001        830508     ED Strap top 3p               Top   
 83527   812371001        812371    Strap top 3-pack          Vest top   
 28591   623522008        623522          Nina Strap          Vest top   
 72495   767869001        767869   V-neck Strap Top.          Vest top   
 62642   736870017        736870    Strap Top 2 pack          Vest top   
 9036    528673003        528673     Gabby strap top          Vest top   
 34

In [12]:
test_article_ids = [
    108775015,  # Strap top
    681107007,  # Dress
    686284001,  # Sweater
    754256001,  # Bra
    755712001   # Shirt
]

for article_id in test_article_ids:
    query_product, recommendations = get_similar_products_text(
        article_id=article_id,
        top_k=5
    )

    print("\nQUERY PRODUCT")
    display(query_product)

    print("RECOMMENDATIONS")
    display(recommendations)


QUERY PRODUCT


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
0,108775015,108775,Strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic


RECOMMENDATIONS


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
10109,538699001,538699,V-neck strap top,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.657127
77230,788464001,788464,Chloe Dress,Dress,Dark Red,Womens Everyday Collection,Jersey Fancy,0.451260
35305,646265001,646265,ES Alexandra strap,Vest top,Black,Womens Trend,Jersey Fancy,0.430839
87915,830508001,830508,ED Strap top 3p,Top,Black,H&M+,Jersey Fancy,0.418206
83527,812371001,812371,Strap top 3-pack,Vest top,Black,Womens Everyday Basics,Jersey Basic,0.411809



QUERY PRODUCT


...

,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
64526,742079002,742079,Panorama mid support bra,Bra,Grey,Ladies H&M Sport,Jersey Fancy,0.621652
56020,712711001,712711,Greenville medium support bra,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.584278
34107,640552001,640552,GREENVILLE med support spor,Bra,Black,Ladies H&M Sport,Jersey Fancy,0.547654
91832,851374004,851374,Karin mid support bra,Top,Yellow,Ladies H&M Sport,Jersey Fancy,0.536967
3183,449744020,449744,Karin Medium Support Bra,Bra,Dark Blue,Ladies H&M Sport,Jersey Fancy,0.525351



QUERY PRODUCT


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name
68888,755712001,755712,Sune slipover set,Shirt,White,Kids Boy,Shirts


RECOMMENDATIONS


,article_id,product_code,prod_name,product_type_name,colour_group_name,section_name,garment_group_name,cosine_similarity
6243,503834001,503834,Sune slipover set,Shirt,Blue,Kids Boy,Shirts,0.678097
66017,746313002,746313,TVP Sven,Garment Set,Dark Red,Baby Boy,Woven/Jersey/Knitted mix Baby,0.507886
89547,838804001,838804,Lenny set,Other accessories,Blue,Baby Essentials & Complements,Accessories,0.501766
75011,779420001,779420,Jacob Cardigan set,Shirt,Dark Blue,Kids Boy,Shirts,0.492434
96987,872483001,872483,Sune WD shirt (TVP),Shirt,White,Kids Boy,Shirts,0.490845


## Initial Findings

The text-based similarity baseline captures fine-grained product relationships that are not visible through categorical metadata alone.

It performs especially well when product names and descriptions contain strong semantic signals, such as:

- similar product family names
- repeated style terms
- product-specific wording in descriptions

Examples include:
- `Dahlia Dress` retrieving other Dahlia-related dresses
- `GREENVILLE high support bra` retrieving semantically similar support bras

However, the text-only model also shows clear limitations:

- It can retrieve products from less relevant categories when generic words dominate the representation.
- Terms such as `strap`, `set`, or seasonal/product-name tokens may create misleading matches.
- Additional filtering is needed to avoid recommending:
  - the same product in different colour variants
  - multiple colour variants of the same recommended product

To address this, the recommendation function filters results by `product_code` so that each product family appears at most once.

Overall, text-based similarity improves fine-grained matching, but it should be combined with categorical information to produce more reliable similar-product recommendations.